# Annual Riparian Land Cover Indices and Water Quality Measurements from Lake Baringo, Kenya (2019–2026) Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All dataset record sets, fields, and columns are referenced strictly by their `@id` identifiers, as required for robust and reproducible exploration.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.wdkf-r3se/fair2.json'  # Provided URL

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata object and print the name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities are referenced by their `@id`. We will enumerate the record sets and fields to inspect what is available in the dataset.

In [ ]:
# List available record sets and their @id
print("Record sets available in the dataset:")
for rs in dataset.record_sets:
    print(f"  @id: {rs['@id']} | Name: {rs.get('name', '')}")

# List fields for each record set, referenced by their @id
for rs in dataset.record_sets:
    print(f"\nFields in record set {rs['@id']}:")
    fields = rs.get('field', [])
    # Ensure fields is a list
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        print(f"   Field @id: {f['@id']} | Name: {f.get('name', '')}")

## 3. Data Extraction
Load data from the main record sets (referenced by their `@id`) into Pandas DataFrames for further analysis.

Below, data from each record set is loaded. Use the `@id` value to reference each entity explicitly.

In [ ]:
# Find all record set @id identifiers
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = dict()

# Load records for each record set using its @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show column names (fields) for the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Columns (@ids) in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    print("\nPreviewing first 5 records:")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing, removing outliers, transforming distributions, or grouping data by key attributes. All columns and fields are referenced using their `@id`.

Below, we select a numeric field, filter values by threshold, normalize, and group by a categorical field if available.

In [ ]:
# Choose the main record set and numeric field by @id
rs_ids = record_set_ids

if rs_ids:
    record_set_id = rs_ids[0]  # Example: use the first record set
    df = dataframes[record_set_id]

    # List columns to find numeric fields (@id)
    print("Available field columns (@id):", df.columns.tolist())

    # Attempt to select a numeric field by @id (example: NDVI index or water quality numeric field)
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean()
        # Filter records above threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records in {record_set_id} with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        field_norm = f"{numeric_field}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalization of {numeric_field}:")
        display(filtered_df[[numeric_field, field_norm]].head())

        # Attempt to group by a categorical field if present
        categoricals = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        if categoricals:
            group_field = categoricals[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"\nGrouped data by {group_field}:")
                display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using pandas or matplotlib.

In [ ]:
import matplotlib.pyplot as plt

# Visualization: Histogram and scatter plot for numerical and categorical fields
if rs_ids and numeric_fields and categoricals:
    # Histogram for the normalized field
    filtered_df[field_norm].hist(bins=30)
    plt.title(f"Distribution of {field_norm} in filtered records")
    plt.xlabel(field_norm)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot for numeric vs. categorical (if applicable)
    plt.figure(figsize=(8,5))
    plt.scatter(filtered_df[group_field], filtered_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field} in filtered data")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook used `mlcroissant` to load, inspect, and process the FAIR^2 riparian land cover and water quality dataset. Using strict `@id` referencing, analysis and visualizations were performed to reveal patterns and distributions in physical land cover indices and station-level measurements. Further data modeling can leverage this structure for machine learning or environmental research.